In [94]:
import pandas as pd
import pandas as pd
import importlib
import utils   
importlib.reload(utils)
from utils import *
import ast
import pandas as pd
import pandas as pd
import os

# pd.set_option("display.max_rows", None)          # show all rows
# pd.set_option("display.max_columns", None)       # show all columns
# pd.set_option("display.max_colwidth", None)      # show full text in each cell
# pd.set_option("display.width", None)             # use full available width
# pd.set_option("display.expand_frame_repr", False)  # avoid wrapping into multiple blocks

In [95]:
import os
import pandas as pd

# tool names mapped to actual file names
tool_to_file = {
    "vllm": "vllm_final_df.csv",
    "sglang": "sglang_final_df.csv",
    "tensorrt_llm": "tensorrt_llm_final_df.csv",
    "lmdeploy": "lmdeploy_final_df.csv",
    "flashinfer": "flashinfer_final_df.csv",
}

base_dir = "/Users/forough/VScodeProjects/VScode_serving/dataset/merged_used_imports_params"

# store each dataframe separately
dfs = {}

for tool_name, file_name in tool_to_file.items():
    path = os.path.join(base_dir, file_name)

    df = pd.read_csv(path).sort_values(by="repo_full_name").reset_index(drop=True)
    df = df.fillna("")

    # optional: add tool_name column inside each dataframe
    df["tool_name"] = tool_name

    dfs[tool_name] = df


# Step 1: get unique repo_full_name and where they appeared
repo_to_tools = {}

for tool_name, df in dfs.items():
    if "repo_full_name" not in df.columns:
        continue

    unique_repos = (
        df["repo_full_name"]
        .astype(str)
        .str.strip()
    )
    unique_repos = unique_repos[unique_repos != ""].unique()

    for repo in unique_repos:
        if repo not in repo_to_tools:
            repo_to_tools[repo] = []
        repo_to_tools[repo].append(tool_name)


# Step 2: store in dataframe
result_df = pd.DataFrame(
    [
        {
            "repo_full_name": repo,
            "appeared in": sorted(tools)
        }
        for repo, tools in repo_to_tools.items()
    ]
).sort_values(by="repo_full_name").reset_index(drop=True)

result_df

,repo_full_name,appeared in
0,0russwest0/Agent-R1,[vllm]
1,0xLDF/Seg2Any,[vllm]
2,0xk1h0/sven_modified,[vllm]
3,18907305772/FuseAI,[vllm]
4,1Wata/mllmTracker,[vllm]
5,1a3orn/grpo-changes,[vllm]
6,2noise/ChatTTS,[vllm]
7,46319943/DianpingAnalyzer,[vllm]
8,666xz666/CoH-Impl,[vllm]
9,A-baoYang/LLM-Finetune-Guide,[vllm]


In [96]:
# keep only rows where repo appears in at least 2 tools
result_df_multi = result_df[result_df["appeared in"].apply(lambda x: len(x) >= 2)].copy()

print(len(result_df_multi))
result_df_multi

33


,repo_full_name,appeared in
43,Andrew-Zhang/vllm-context-shrinking-scratch,"[flashinfer, vllm]"
147,EvolvingLMMs-Lab/lmms-eval,"[sglang, vllm]"
156,FlagAI-Open/OpenSeek,"[sglang, vllm]"
182,HKUNLP/ChunkLlama,"[flashinfer, vllm]"
210,IBM/vllm,"[flashinfer, vllm]"
222,Ja1Zhou/vllm,"[flashinfer, vllm]"
267,LMCache/LMCache,"[flashinfer, vllm]"
299,MananAg007/diverse-data-synthesis,"[flashinfer, vllm]"
334,NVIDIA/TensorRT-LLM,"[flashinfer, tensorrt_llm]"
335,NVIDIA/TensorRT-Model-Optimizer,"[sglang, tensorrt_llm, vllm]"


In [97]:
# helper
def is_nonempty_cell(x):
    if pd.isna(x):
        return False
    if isinstance(x, str):
        return x.strip() != ""
    return True

# make a fresh copy
result_df_multi = result_df_multi.copy()

# recompute for the CURRENT rows only
non_empty_columns_all = []

for _, row in result_df_multi.iterrows():
    repo = row["repo_full_name"]
    appeared_tools = row["appeared in"]   # assumed to be a list or tuple

    non_empty_cols = set()

    for tool_name in appeared_tools:
        df = dfs[tool_name]

        sub_df = df[df["repo_full_name"].astype(str).str.strip() == str(repo).strip()]

        if sub_df.empty:
            continue

        cols_to_check = [c for c in df.columns if c not in ["repo_full_name", "tool_name"]]

        for col in cols_to_check:
            if sub_df[col].apply(is_nonempty_cell).any():
                non_empty_cols.add(col)

    non_empty_columns_all.append(sorted(non_empty_cols))

# now lengths match
result_df_multi["non_empty_columns"] = non_empty_columns_all

# print(result_df_multi_with_cols)



In [98]:
result_df_multi_with_cols = result_df_multi.copy()
result_df_multi_with_cols["non_empty_columns"] = non_empty_columns_all
result_df_multi_with_cols

,repo_full_name,appeared in,non_empty_columns
43,Andrew-Zhang/vllm-context-shrinking-scratch,"[flashinfer, vllm]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_topk, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]"
147,EvolvingLMMs-Lab/lmms-eval,"[sglang, vllm]","[sglang_parallelism, vllm_parallelism]"
156,FlagAI-Open/OpenSeek,"[sglang, vllm]","[sglang_Continuous batching, sglang_parallelism, vllm_parallelism]"
182,HKUNLP/ChunkLlama,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_page, flashinfer_sampling, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Prefix caching, vllm_Quantization, vllm_parallelism]"
210,IBM/vllm,"[flashinfer, vllm]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_topk, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_parallelism]"
222,Ja1Zhou/vllm,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_norm, flashinfer_page, flashinfer_sampling, flashinfer_topk, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]"
267,LMCache/LMCache,"[flashinfer, vllm]","[flashinfer_page, flashinfer_sparse, vllm_KVCaching, vllm_Prefix caching, vllm_parallelism]"
299,MananAg007/diverse-data-synthesis,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_page, flashinfer_quantization, vllm_Multi-LoRA, vllm_Prefix caching, vllm_parallelism]"
334,NVIDIA/TensorRT-LLM,"[flashinfer, tensorrt_llm]","[flashinfer_activation, flashinfer_norm, flashinfer_quantization, flashinfer_rope, tensorrt_llm_Guided Decoding, tensorrt_llm_In-Flight Batching, tensorrt_llm_KV Cache Management, tensorrt_llm_LoRA Support, tensorrt_llm_Paged Attention, tensorrt_llm_Quantization, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative decoding]"
335,NVIDIA/TensorRT-Model-Optimizer,"[sglang, tensorrt_llm, vllm]","[sglang_Quantization, tensorrt_llm_Continuous batching, vllm_Quantization]"


In [99]:
pd.set_option("display.max_rows", None)          # show all rows
pd.set_option("display.max_columns", None)       # show all columns
pd.set_option("display.max_colwidth", None)      # show full text in each cell
pd.set_option("display.width", None)             # use full available width
pd.set_option("display.expand_frame_repr", False)  # avoid wrapping into multiple blocks
def normalize_appeared_in(x):
    if isinstance(x, str):
        parts = [s.strip() for s in x.split(",") if s.strip()]
        return sorted(parts)
    return sorted(list(x))

result_df_multi_with_cols["appeared in"] = result_df_multi_with_cols["appeared in"].apply(normalize_appeared_in)
result_df_multi_with_cols["_appeared_in_key"] = result_df_multi_with_cols["appeared in"].apply(lambda x: ", ".join(x))

result_df_multi_with_cols = (
    result_df_multi_with_cols
    .sort_values(by="_appeared_in_key")
    .drop(columns="_appeared_in_key")
    .reset_index(drop=True)
)


result_df_multi_with_cols

,repo_full_name,appeared in,non_empty_columns
0,sgl-project/sglang,"[flashinfer, sglang]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Multi-LoRA, sglang_all parallelism, sglang_kv_cache]"
1,qingshanlanshan/SGL-KVS,"[flashinfer, sglang]","[flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]"
2,pkking/sglang,"[flashinfer, sglang]","[flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]"
3,zejia-lin/Bullet,"[flashinfer, sglang]","[flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]"
4,NVIDIA/TensorRT-LLM,"[flashinfer, tensorrt_llm]","[flashinfer_activation, flashinfer_norm, flashinfer_quantization, flashinfer_rope, tensorrt_llm_Guided Decoding, tensorrt_llm_In-Flight Batching, tensorrt_llm_KV Cache Management, tensorrt_llm_LoRA Support, tensorrt_llm_Paged Attention, tensorrt_llm_Quantization, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative decoding]"
5,Andrew-Zhang/vllm-context-shrinking-scratch,"[flashinfer, vllm]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_topk, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]"
6,vllm-project/vllm,"[flashinfer, vllm]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_topk, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_parallelism]"
7,swjeong9/test-repo,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_norm, flashinfer_page, flashinfer_sampling, vllm_CUDA, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_parallelism]"
8,snowflakedb/ArcticInference,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_logits_processor, flashinfer_norm, flashinfer_quantization, flashinfer_sampling, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]"
9,sjtu-zwh/vllm,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_norm, flashinfer_page, flashinfer_sampling, vllm_CUDA, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_parallelism]"


In [100]:
import ast
import numpy as np
import pandas as pd

def normalize_as_sorted_tuple(x):
    """
    Convert each cell into a sorted tuple so item order does not matter.
    Works for:
    - lists
    - tuples
    - sets
    - numpy arrays
    - comma-separated strings
    - string-form lists
    """

    # Case 1: value is already list-like
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return tuple(sorted(str(item).strip() for item in x if str(item).strip()))

    # Case 2: missing value
    if pd.isna(x):
        return tuple()

    # Case 3: string value
    if isinstance(x, str):
        x = x.strip()

        # Case 3a: string-form list, e.g. "['vLLM', 'SGLang']"
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, (list, tuple, set, np.ndarray)):
                return tuple(sorted(str(item).strip() for item in parsed if str(item).strip()))
        except Exception:
            pass

        # Case 3b: comma-separated string, e.g. "vLLM, SGLang"
        return tuple(sorted(item.strip() for item in x.split(",") if item.strip()))

    # Case 4: any other value
    return tuple([str(x).strip()])


# Make a copy without repo_full_name
unique_df = result_df_multi_with_cols.drop(columns=["repo_full_name"]).copy()

# Create temporary comparison keys
unique_df["_appeared_in_key"] = unique_df["appeared in"].apply(normalize_as_sorted_tuple)
unique_df["_non_empty_columns_key"] = unique_df["non_empty_columns"].apply(normalize_as_sorted_tuple)

# Remove duplicated rows based on appeared in + non_empty_columns
# Order of items inside both columns is ignored
unique_df = (
    unique_df
    .drop_duplicates(
        subset=["_appeared_in_key", "_non_empty_columns_key"],
        keep="first"
    )
    .drop(columns=["_appeared_in_key", "_non_empty_columns_key"])
    .reset_index(drop=True)
)


# Create temporary key for sorting
unique_df["_appeared_in_sort_key"] = unique_df["appeared in"].apply(
    lambda x: ", ".join(normalize_as_sorted_tuple(x))
)

# Sort rows based on appeared in
unique_df = (
    unique_df
    .sort_values(by="_appeared_in_sort_key")
    .drop(columns=["_appeared_in_sort_key"])
    .reset_index(drop=True)
)

unique_df

for i, row in unique_df.iterrows():
    print("row number is:", i)
    print("appeared in is:", row['appeared in'])
    print("methods: ", row['non_empty_columns'])
    print("\n")

row number is: 0
appeared in is: ['flashinfer', 'sglang']
methods:  ['flashinfer_Continuous batching', 'flashinfer_FlashInfer Attention Kernels', 'flashinfer_cascade', 'flashinfer_comm', 'flashinfer_flashinfer.gemm', 'flashinfer_fused_moe', 'flashinfer_norm', 'flashinfer_page', 'flashinfer_quantization', 'flashinfer_topk', 'sglang_Multi-LoRA', 'sglang_all parallelism', 'sglang_kv_cache']


row number is: 1
appeared in is: ['flashinfer', 'sglang']
methods:  ['flashinfer_FlashInfer Attention Kernels', 'flashinfer_cascade', 'flashinfer_flashinfer.gemm', 'flashinfer_fused_moe', 'flashinfer_norm', 'flashinfer_page', 'flashinfer_quantization', 'flashinfer_topk', 'sglang_Quantization', 'sglang_all parallelism', 'sglang_kv_cache']


row number is: 2
appeared in is: ['flashinfer', 'tensorrt_llm']
methods:  ['flashinfer_activation', 'flashinfer_norm', 'flashinfer_quantization', 'flashinfer_rope', 'tensorrt_llm_Guided Decoding', 'tensorrt_llm_In-Flight Batching', 'tensorrt_llm_KV Cache Management

In [101]:
row_0 = ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'sglang_Request Scheduling methods', 'sglang_Parallel Computation methods', 'sglang_Memory Management methods'],
row_1 = ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'sglang_Low-bit Quantization methods', 'sglang_Parallel Computation methods', 'sglang_Memory Management methods'],
row_2 = ['flashinfer_Kernel Fusion', 'flashinfer_Kernel Fusion', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Additional Kernel Optimization Methods', 'tensorrt_llm_Decoding Algorithm methods', 'tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Networking Pruning', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Low-bit Quantization methods', 'tensorrt_llm_Speculative Decoding', 'tensorrt_llm_Speculative Decoding'],
row_3 = ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
row_4 = ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
row_5 = ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Quantization', 'vllm_Parallel Computation methods']

row_6 = ['flashinfer_Memory Management methods', 'flashinfer_Attention Simplification', 'vllm_Memory Management methods', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods']

row_7 = ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'vllm_Network Pruning', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods']

row_8 = ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods']

row_9 = ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods']

row_10 = ['flashinfer_Tailored Attention', 'flashinfer_Attention Simplification', 'vllm_Network Pruning']

row_11 = ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods']

row_12 = ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods']

row_13 = ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods']

row_14 = ['flashinfer_Kernel Fusion', 'vllm_Network Pruning']

row_15 = ['flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods']

row_16 = ['flashinfer_Tailored Attention', 'flashinfer_Kernel Fusion', 'flashinfer_Kernel Fusion', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods']

row_17 = ['flashinfer_Tailored Attention', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods']

row_18 = ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods']

row_19 = ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods']

row_20 = ['lmdeploy_Memory Management methods', 'sglang_Memory Management methods', 'sglang_UNMAPPED', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods']

row_21 = ['lmdeploy_Memory Management methods', 'lmdeploy_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods']

row_22 = ['sglang_Low-bit Quantization methods', 'tensorrt_llm_Request Scheduling methods', 'vllm_Quantization']

row_23 = ['sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods']

row_24 = ['sglang_Parallel Computation methods', 'vllm_Parallel Computation methods']

row_25 = ['sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'vllm_Parallel Computation methods']

row_26 = ['sglang_Memory Management methods', 'sglang_Parallel Computation methods', 'vllm_Parallel Computation methods']

row_27 = ['tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Parallel Computation methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods']

row_28 = ['tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Parallel Computation methods', 'tensorrt_llm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods']

In [102]:
mapped_methods = [
    ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'sglang_Request Scheduling methods', 'sglang_Parallel Computation methods', 'sglang_Memory Management methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'sglang_Low-bit Quantization methods', 'sglang_Parallel Computation methods', 'sglang_Memory Management methods'],
    ['flashinfer_Kernel Fusion', 'flashinfer_Kernel Fusion', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Additional Kernel Optimization Methods', 'tensorrt_llm_Decoding Algorithm methods', 'tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Networking Pruning', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Low-bit Quantization methods', 'tensorrt_llm_Speculative Decoding', 'tensorrt_llm_Speculative Decoding'],
    ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['flashinfer_Memory Management methods', 'flashinfer_Attention Simplification', 'vllm_Memory Management methods', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'vllm_Network Pruning', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Attention Simplification', 'vllm_Network Pruning'],
    ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods'],
    ['flashinfer_Kernel Fusion', 'vllm_Network Pruning'],
    ['flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Low-bit Quantization methods', 'flashinfer_Kernel Fusion', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Kernel Fusion', 'flashinfer_Kernel Fusion', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods'],
    ['flashinfer_Tailored Attention', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_UNMAPPED', 'vllm_Kernel Optimizations methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['flashinfer_UNMAPPED', 'flashinfer_Tailored Attention', 'flashinfer_Cascade Inference', 'flashinfer_Parallel Computation methods', 'flashinfer_Kernel Fusion', 'flashinfer_Conditional Computing', 'flashinfer_Kernel Fusion', 'flashinfer_Memory Management methods', 'flashinfer_Low-bit Quantization methods', 'flashinfer_UNMAPPED', 'flashinfer_Kernel Fusion', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Network Pruning', 'vllm_Quantization', 'vllm_Speculative Decoding', 'vllm_Parallel Computation methods'],
    ['lmdeploy_Memory Management methods', 'sglang_Memory Management methods', 'sglang_UNMAPPED', 'vllm_Memory Management methods', 'vllm_Parallel Computation methods'],
    ['lmdeploy_Memory Management methods', 'lmdeploy_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['sglang_Low-bit Quantization methods', 'tensorrt_llm_Request Scheduling methods', 'vllm_Quantization'],
    ['sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'vllm_Network Pruning', 'vllm_Parallel Computation methods'],
    ['sglang_Parallel Computation methods', 'vllm_Parallel Computation methods'],
    ['sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'sglang_Parallel Computation methods', 'vllm_Parallel Computation methods'],
    ['sglang_Memory Management methods', 'sglang_Parallel Computation methods', 'vllm_Parallel Computation methods'],
    ['tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Parallel Computation methods', 'vllm_Request Scheduling methods', 'vllm_Memory Management methods', 'vllm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods'],
    ['tensorrt_llm_Request Scheduling methods', 'tensorrt_llm_Memory Management methods', 'tensorrt_llm_Parallel Computation methods', 'tensorrt_llm_Memory Management methods', 'vllm_Quantization', 'vllm_Parallel Computation methods']
]

# Check that the number of lists matches the number of rows
assert len(mapped_methods) == len(unique_df), f"Length mismatch: {len(mapped_methods)} values for {len(unique_df)} rows"

# Add as a new column
unique_df["mapped_methods"] = mapped_methods

In [103]:
unique_df

,appeared in,non_empty_columns,mapped_methods
0,"[flashinfer, sglang]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Multi-LoRA, sglang_all parallelism, sglang_kv_cache]","[flashinfer_UNMAPPED, flashinfer_Tailored Attention, flashinfer_Cascade Inference, flashinfer_Parallel Computation methods, flashinfer_Kernel Fusion, flashinfer_Conditional Computing, flashinfer_Kernel Fusion, flashinfer_Memory Management methods, flashinfer_Low-bit Quantization methods, flashinfer_Kernel Fusion, sglang_Request Scheduling methods, sglang_Parallel Computation methods, sglang_Memory Management methods]"
1,"[flashinfer, sglang]","[flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]","[flashinfer_Tailored Attention, flashinfer_Cascade Inference, flashinfer_Kernel Fusion, flashinfer_Conditional Computing, flashinfer_Kernel Fusion, flashinfer_Memory Management methods, flashinfer_Low-bit Quantization methods, flashinfer_Kernel Fusion, sglang_Low-bit Quantization methods, sglang_Parallel Computation methods, sglang_Memory Management methods]"
2,"[flashinfer, tensorrt_llm]","[flashinfer_activation, flashinfer_norm, flashinfer_quantization, flashinfer_rope, tensorrt_llm_Guided Decoding, tensorrt_llm_In-Flight Batching, tensorrt_llm_KV Cache Management, tensorrt_llm_LoRA Support, tensorrt_llm_Paged Attention, tensorrt_llm_Quantization, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative decoding]","[flashinfer_Kernel Fusion, flashinfer_Kernel Fusion, flashinfer_Low-bit Quantization methods, flashinfer_Additional Kernel Optimization Methods, tensorrt_llm_Decoding Algorithm methods, tensorrt_llm_Request Scheduling methods, tensorrt_llm_Memory Management methods, tensorrt_llm_Networking Pruning, tensorrt_llm_Memory Management methods, tensorrt_llm_Low-bit Quantization methods, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative Decoding]"
3,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_page, flashinfer_sampling, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Prefix caching, vllm_Quantization, vllm_parallelism]","[flashinfer_Tailored Attention, flashinfer_Memory Management methods, flashinfer_UNMAPPED, vllm_Kernel Optimizations methods, vllm_Request Scheduling methods, vllm_Memory Management methods, vllm_Network Pruning, vllm_Memory Management methods, vllm_Quantization, vllm_Parallel Computation methods]"
4,"[flashinfer, vllm]","[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_topk, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Quantization, vllm_parallelism]","[flashinfer_UNMAPPED, flashinfer_Tailored Attention, flashinfer_Cascade Inference, flashinfer_Parallel Computation methods, flashinfer_Kernel Fusion, flashinfer_Conditional Computing, flashinfer_Kernel Fusion, flashinfer_Memory Management methods, flashinfer_Low-bit Quantization methods, flashinfer_UNMAPPED, flashinfer_Kernel Fusion, vllm_Kernel Optimizations methods, vllm_Request Scheduling methods, vllm_Memory Management methods, vllm_Network Pruning, vllm_Quantization, vllm_Parallel Computation methods]"
5,"[flashinfer, vllm]","[flashinfer_FlashInfer Attention Kernels, flashinfer_page, flashinfer_sampling, vllm_CUDA, vllm_Chunked prefill, vllm_Quantization, vllm_parallelism]","[flashinfer_Tailored Attention, flashinfer_Memory Management methods, flashinfer_UNMAPPED, vllm_Kernel Optimizations methods, vllm_Request 

In [105]:
import ast
import pandas as pd

df = result_df_multi_with_cols.copy()

def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, tuple) or isinstance(x, set):
        return list(x)
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, tuple, set)):
                return list(val)
        except Exception:
            pass
        return [x]
    return []

# make sure both columns are real lists
df["appeared in"] = df["appeared in"].apply(lambda x: sorted(ensure_list(x)))
df["non_empty_columns"] = df["non_empty_columns"].apply(ensure_list)

# key for grouping
df["appeared_in_key"] = df["appeared in"].apply(tuple)

# collect all unique values from non_empty_columns for each unique appeared in
grouped = (
    df.groupby("appeared_in_key", as_index=False)
      .agg({
          "repo_full_name": "count",   # optional: number of rows in this group
          "non_empty_columns": lambda s: sorted(set(v for lst in s for v in lst))
      })
      .rename(columns={"repo_full_name": "count of unique respositories"})
)

# convert tuple key back to list if you want
grouped["appeared in"] = grouped["appeared_in_key"].apply(list)
grouped = grouped.drop(columns="appeared_in_key")

# optional column order
grouped = grouped[["appeared in", "count of unique respositories", "non_empty_columns"]]


grouped = grouped.sort_values(
    by="count of unique respositories",
    ascending=False
)

grouped

,appeared in,count of unique respositories,non_empty_columns
2,"[flashinfer, vllm]",18,"[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_green_ctx, flashinfer_logits_processor, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_sparse, flashinfer_topk, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Prefix caching, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]"
6,"[sglang, vllm]",5,"[sglang_Continuous batching, sglang_all parallelism, sglang_data parallelism, sglang_parallelism, sglang_pipeline parallelism, sglang_tensor parallelism, vllm_Multi-LoRA, vllm_parallelism]"
0,"[flashinfer, sglang]",4,"[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Multi-LoRA, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]"
7,"[tensorrt_llm, vllm]",2,"[tensorrt_llm_Chunked prefill, tensorrt_llm_Continuous batching, tensorrt_llm_KV Cache Management, tensorrt_llm_Multi-GPU Multi-Node, tensorrt_llm_Paged Attention, tensorrt_llm_parallelism, vllm_Chunked prefill, vllm_Continuous batching, vllm_Prefix caching, vllm_Quantization, vllm_parallelism]"
1,"[flashinfer, tensorrt_llm]",1,"[flashinfer_activation, flashinfer_norm, flashinfer_quantization, flashinfer_rope, tensorrt_llm_Guided Decoding, tensorrt_llm_In-Flight Batching, tensorrt_llm_KV Cache Management, tensorrt_llm_LoRA Support, tensorrt_llm_Paged Attention, tensorrt_llm_Quantization, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative decoding]"
3,"[lmdeploy, sglang, vllm]",1,"[lmdeploy_Prefix caching, sglang_Prefix caching, sglang_structured outputs, vllm_Prefix caching, vllm_parallelism]"
4,"[lmdeploy, vllm]",1,"[lmdeploy_Continuous batching, lmdeploy_Prefix caching, vllm_Quantization, vllm_parallelism]"
5,"[sglang, tensorrt_llm, vllm]",1,"[sglang_Quantization, tensorrt_llm_Continuous batching, vllm_Quantization]"


In [106]:
# # Desired order based on your attached table
# desired_order = [
#     ("sglang", "vllm"),                         # 1. vLLM + SGLang
#     ("lmdeploy", "vllm"),                       # 2. vLLM + LMDeploy
#     ("flashinfer", "vllm"),                     # 3. vLLM + FlashInfer
#     ("sglang", "tensorrt_llm", "vllm"),         # 4. vLLM + SGLang + TensorRT-LLM
#     ("flashinfer", "sglang"),                   # 5. SGLang + FlashInfer
#     ("tensorrt_llm", "vllm"),                  # 6. vLLM + TensorRT-LLM
#     ("flashinfer", "tensorrt_llm"),            # 7. TensorRT-LLM + FlashInfer
#     ("lmdeploy", "sglang", "vllm")             # 8. vLLM + SGLang + LMDeploy
# ]

# # Create ranking dictionary
# order_map = {
#     combo: i for i, combo in enumerate(desired_order)
# }

# # Create a sortable key from appears_in
# grouped["appears_in_key"] = grouped["appeared in"].apply(lambda x: tuple(sorted(x)))

# # Assign row order based on desired_order
# grouped["custom_order"] = grouped["appears_in_key"].map(order_map)

# # Sort according to the table order
# grouped = grouped.sort_values("custom_order")

# # Remove helper columns
# grouped = grouped[["appeared in", "count of unique respositories"]]

# print(grouped)

In [107]:
import ast
import pandas as pd

# Make a copy so the original dataframe is not modified
df = grouped.copy()

# Total number of repositories for each framework
framework_totals = {
    "vllm": 1821,
    "sglang": 54,
    "tensorrt_llm": 15,
    "lmdeploy": 45,
    "flashinfer": 52
}

# Desired order based on your table
desired_order = [
    ("sglang", "vllm"),                         # vLLM + SGLang
    ("lmdeploy", "vllm"),                       # vLLM + LMDeploy
    ("flashinfer", "vllm"),                     # vLLM + FlashInfer
    ("sglang", "tensorrt_llm", "vllm"),         # vLLM + SGLang + TensorRT-LLM
    ("flashinfer", "sglang"),                   # SGLang + FlashInfer
    ("tensorrt_llm", "vllm"),                   # vLLM + TensorRT-LLM
    ("flashinfer", "tensorrt_llm"),             # TensorRT-LLM + FlashInfer
    ("lmdeploy", "sglang", "vllm")              # vLLM + SGLang + LMDeploy
]

order_map = {combo: i for i, combo in enumerate(desired_order)}

# Convert appeared in column safely to list if it is stored as a string
def convert_to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df["appeared in"] = df["appeared in"].apply(convert_to_list)

# Create sorting key
df["appears_in_key"] = df["appeared in"].apply(lambda x: tuple(sorted(x)))

# Add custom order
df["custom_order"] = df["appears_in_key"].map(order_map)

# Keep only rows that exist in your desired_order
df = df[df["custom_order"].notna()]

# Sort based on your table order
df = df.sort_values("custom_order")

# Add percentage in front of each framework name
def add_percentage(row):
    frameworks = row["appeared in"]
    count = row["count of unique respositories"]  # keep your original column name

    formatted = []

    for framework in frameworks:
        total = framework_totals[framework]
        percentage = (count / total) * 100
        formatted.append(f"{framework} ({percentage:.2f}%)")

    return formatted

df["appeared in"] = df.apply(add_percentage, axis=1)

# Final dataframe
df = df[["appeared in", "count of unique respositories"]]

print(df.to_string(index=False))

                                         appeared in  count of unique respositories
                      [sglang (9.26%), vllm (0.27%)]                              5
                    [lmdeploy (2.22%), vllm (0.05%)]                              1
                 [flashinfer (34.62%), vllm (0.99%)]                             18
[sglang (1.85%), tensorrt_llm (6.67%), vllm (0.05%)]                              1
                [flashinfer (7.69%), sglang (7.41%)]                              4
               [tensorrt_llm (13.33%), vllm (0.11%)]                              2
          [flashinfer (1.92%), tensorrt_llm (6.67%)]                              1
    [lmdeploy (2.22%), sglang (1.85%), vllm (0.05%)]                              1


In [108]:
def total_unique_repos_union(appeared_in_list, dfs):
    all_repos = set()
    for key in appeared_in_list:
        if key in dfs and isinstance(dfs[key], pd.DataFrame):
            if "repo_full_name" in dfs[key].columns:
                repos = dfs[key]["repo_full_name"].dropna().astype(str)
                all_repos.update(repos)
    return len(all_repos)

grouped["total_unique repos that used appeared in frameworks (no repetitive)"] = grouped["appeared in"].apply(
    lambda lst: total_unique_repos_union(lst, dfs)
)
grouped

,appeared in,count of unique respositories,non_empty_columns,total_unique repos that used appeared in frameworks (no repetitive)
2,"[flashinfer, vllm]",18,"[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_green_ctx, flashinfer_logits_processor, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_sampling, flashinfer_sparse, flashinfer_topk, vllm_CUDA, vllm_Chunked prefill, vllm_KVCaching, vllm_Multi-LoRA, vllm_Prefix caching, vllm_Quantization, vllm_Speculative decoding, vllm_parallelism]",1245
6,"[sglang, vllm]",5,"[sglang_Continuous batching, sglang_all parallelism, sglang_data parallelism, sglang_parallelism, sglang_pipeline parallelism, sglang_tensor parallelism, vllm_Multi-LoRA, vllm_parallelism]",1231
0,"[flashinfer, sglang]",4,"[flashinfer_Continuous batching, flashinfer_FlashInfer Attention Kernels, flashinfer_cascade, flashinfer_comm, flashinfer_flashinfer.gemm, flashinfer_fused_moe, flashinfer_norm, flashinfer_page, flashinfer_quantization, flashinfer_topk, sglang_Multi-LoRA, sglang_Quantization, sglang_all parallelism, sglang_kv_cache]",45
7,"[tensorrt_llm, vllm]",2,"[tensorrt_llm_Chunked prefill, tensorrt_llm_Continuous batching, tensorrt_llm_KV Cache Management, tensorrt_llm_Multi-GPU Multi-Node, tensorrt_llm_Paged Attention, tensorrt_llm_parallelism, vllm_Chunked prefill, vllm_Continuous batching, vllm_Prefix caching, vllm_Quantization, vllm_parallelism]",1230
1,"[flashinfer, tensorrt_llm]",1,"[flashinfer_activation, flashinfer_norm, flashinfer_quantization, flashinfer_rope, tensorrt_llm_Guided Decoding, tensorrt_llm_In-Flight Batching, tensorrt_llm_KV Cache Management, tensorrt_llm_LoRA Support, tensorrt_llm_Paged Attention, tensorrt_llm_Quantization, tensorrt_llm_Speculative Decoding, tensorrt_llm_Speculative decoding]",43
3,"[lmdeploy, sglang, vllm]",1,"[lmdeploy_Prefix caching, sglang_Prefix caching, sglang_structured outputs, vllm_Prefix caching, vllm_parallelism]",1235
4,"[lmdeploy, vllm]",1,"[lmdeploy_Continuous batching, lmdeploy_Prefix caching, vllm_Quantization, vllm_parallelism]",1230
5,"[sglang, tensorrt_llm, vllm]",1,"[sglang_Quantization, tensorrt_llm_Continuous batching, vllm_Quantization]",1235
